Team Member Contribution Table

| Member      | Contribution |
| ----------- | ----------- |
| Petter      | CS cloud, building applications using docker, FastAPI development and integration with web-client, deploying application on VM.       |
| Kim   | CS Cloud and CS Cloud documentation       |
| Elliot   | Machine learning development       |
| Agust   | Developed web client, integrated FastAPI to communicate with real models, effectively replacing the temporary dummy model        |

# A2 Sprint Report: Machine Learning

**Task:** Train a linear regression model to predict expert movement scores (AimoScore)

## 1. Collect the Data

Dataset: `AimoScore_WeakLink_big_scores.xls`

- **Samples:** 2094 movement assessments
- **Features:** 40 movement deviation measurements
  - 13 Angle Deviations
  - 25 NASM Deviations
  - 2 Time Deviations
- **Target:** AimoScore (expert score, range 0-1)

## 2. Prepare the Data

- Removed `EstimatedScore` column (not a real feature)
- Split data: 80% training (1675 samples), 20% test (419 samples)


## 3. Select the Model

**Model:** Linear Regression (sklearn `LinearRegression`)

Linear regression maps the 40 features to the expert score:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_{40} x_{40}$$

## 4 & 5. Train and Evaluate Model Variants

We trained 5 variants and evaluated using R², RMSE, and MAE on the test set.

### Variant 1: Baseline
Standard linear regression on all 40 features.

### Variant 2: Feature Selection (p-value)
Used t-test to identify statistically significant features (p < 0.05). Removed 20 features, kept 20.  

R² = 0.5977  
RMSE = 0.1550  
MAE = 0.1167  

### Variant 3: Combined Features (Interaction)
Added one interaction term: `No_12_NASM_Deviation × No_11_NASM_Deviation` (the two features most correlated with target). Total 41 features.  
R² = 0.6412  
RMSE = 0.1464  
MAE = 0.1112  
             
### Variant 4: Outlier Removal (Studentized Residuals)
Removed 83 training samples (5%) with |studentized residual| > 2.  
R² = 0.6153  
RMSE = 0.1516  
MAE = 0.1137  


### Variant 5: Weighted Least Squares
Weighted samples by inverse squared residuals — samples with smaller prediction errors got higher weights.  
R² = 0.6164  
RMSE = 0.1513  
MAE = 0.1149  

## Results

| Model | Test R² | Test RMSE | Test MAE |
|-------|---------|-----------|----------|
| Baseline | 0.6164 | 0.1513 | 0.1150 |
| Feature Selection (p-value) | 0.5977 | 0.1550 | 0.1167 |
| **Combined Features (Interaction)** | **0.6412** | **0.1464** | **0.1112** |
| Outlier Removal (Studentized) | 0.6153 | 0.1516 | 0.1137 |
| Weighted Least Squares | 0.6164 | 0.1513 | 0.1149 |

## 6. Tune the Parameters

For the interaction term variant, we selected features based on correlation with target:
- `No_12_NASM_Deviation` (r = 0.49)
- `No_11_NASM_Deviation` (r = 0.46)

The interaction term captures non-linear relationships where the combined effect of both deviations matters.

## 7. Champion Model Selection

**Champion:** Combined Features (Interaction)

- **R² = 0.6412** — explains 64% of variance in expert scores
- **RMSE = 0.1464** — root mean square error
- **MAE = 0.1112** — average absolute error

### Why it won:
The interaction term `No_12_NASM × No_11_NASM` captured a non-linear relationship that baseline couldn't model. When both deviations are high together, the combined effect on the score is greater than either alone.

### Why others didn't win:
- **Feature Selection:** Removed features that contributed useful information together
- **Outlier Removal:** Lost real signal by removing unusual but valid samples
- **WLS:** Data had constant variance, so weighting didn't help

## Iterations

Following the ML process diagram, we iterated between steps 4 (Train) and 5 (Evaluate) for each variant:

1. Baseline → Evaluate → R² = 0.6164
2. Feature Selection → Evaluate → R² = 0.5977 (worse)
3. Combined Features → Evaluate → R² = 0.6412 (best)
4. Outlier Removal → Evaluate → R² = 0.6153 (worse)
5. WLS → Evaluate → R² = 0.6164 (same as baseline)

Selected Combined Features as champion based on highest R².

## Software development ##

### Deployed Client-Server System ###

During this sprint, a client–server system was implemented to make the trained regression model accessible as a remote service. The system allows external clients to submit feature values and receive a predicted expert score in return. This setup was used to test the deployed model independently of the training environment.


#### Usage of the Deployed System

The deployed system is used by sending a request containing a full record of feature values to the backend service. The service processes the input using the currently deployed regression model and returns a predicted expert score.

The primary use cases during this sprint were:
- Manual testing of the deployed model
- Verifying that model predictions match local evaluation results
- Demonstrating the model through a remote web client



#### Client–Server Architecture

The system follows a simple client–server architecture:

- A backend service hosts the trained regression model and exposes a prediction endpoint.
- A web-based client communicates with the backend using HTTP requests.
- The client sends feature values as input and receives the predicted expert score as a response.

This separation allows the regression model to be updated or replaced without changing the client implementation.


#### System Design

The system was designed with simplicity and extensibility in mind. The backend focuses solely on model inference and input validation, while the client is responsible for user interaction and displaying results.

Design decisions made during this sprint include:
- Keeping the API interface minimal (single prediction endpoint)
- Using a stateless backend service
- Structuring the system so that new model versions can be deployed without affecting client usage


#### Implementation Details

The backend service loads the trained regression model at startup and applies it to incoming requests. Feature values are parsed from the request payload and passed directly to the model to compute the expert score.

The web client was implemented as a lightweight remote interface that allows users to manually input feature values and trigger prediction requests. Responses from the backend are displayed directly to the user, making it easy to verify system behavior during development.


### DevOps Process ###

In this sprint, an initial DevOps process was defined to support continuous testing and deployment of new regression model variants. The goal is to enable systematic comparison of model variants and ensure that the deployed service always uses the best-performing model.

#### Version Control ####

All source code is managed using Git and stored in a shared GitLab repository. This includes the web client, backend server, and machine learning service code, ensuring traceability and reproducibility.

#### Automated Testing of Model Variants

A structured process was defined for testing new regression model variants. Each variant is trained and evaluated using the same metrics and evaluation setup. The results are then compared against the currently deployed champion model.

This ensures that model improvements are assessed consistently and reduces the risk of deploying underperforming models.


#### Deployment Strategy

When a regression model variant outperforms the current champion according to predefined evaluation criteria, it is selected for deployment. The deployed service is then updated to use the new model version.

This strategy allows model updates to be introduced incrementally while keeping the client–server system unchanged.


#### Continuous Integration and Deployment (Current State)

While the DevOps pipeline is still under development, the responsibilities and structure have been defined. Model training, evaluation, comparison, and deployment are treated as repeatable steps.

This foundation enables future automation of the entire process, from training new model variants to deploying an updated prediction service.


#### Continuous Integration and Deployment (Current State)

While the DevOps pipeline is still under development, the responsibilities and structure have been defined. Model training, evaluation, comparison, and deployment are treated as repeatable steps.

This foundation enables future automation of the entire process, from training new model variants to deploying an updated prediction service.
